# Aplicación del Modelo Final a Test — Generación del Submission
## Predicción de Popularidad de Canciones de Spotify — Competencia Kaggle

Este notebook reentrena el modelo final (idéntica semilla e hiperparámetros
que `entrenar_modelo_final.ipynb`) y lo aplica sobre el conjunto de test para
generar el archivo de submission a subir a Kaggle.

**Inputs:** `base_train.csv` (para entrenar) y `base_val.csv` (datos de test).
**Output:** `submission.csv`, con columnas `ID` y `track_popularity`.

No incluye análisis exploratorio ni comparación de modelos — únicamente el
código necesario para reproducir el archivo enviado a la competencia.


## 1. Imports y configuración

**Nota de reproducibilidad:** se fija `scikit-learn==1.6.1`, la misma versión
utilizada en `entrenar_modelo_final.ipynb`, para garantizar que el modelo
reentrenado en este notebook sea numéricamente idéntico al documentado allí.


In [17]:
import subprocess, sys
subprocess.check_call([sys.executable,"-m","pip","install","scikit-learn==1.6.1","-q","--break-system-packages"],
                      stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
for p in ["pandas","numpy"]:
    subprocess.check_call([sys.executable,"-m","pip","install",p,"-q","--break-system-packages"],
                          stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

from sklearn.impute import SimpleImputer
from sklearn.ensemble import ExtraTreesRegressor

import sklearn
print(f"scikit-learn version: {sklearn.__version__}")

SEED = 42


scikit-learn version: 1.6.1


## 2. Carga de datos

In [18]:
train = pd.read_csv('base_train.csv', index_col=0)
test  = pd.read_csv('base_val.csv',   index_col=0)

print(f"Train: {train.shape[0]} filas")
print(f"Test:  {test.shape[0]} filas")


Train: 26266 filas
Test:  6567 filas


## 3. Feature Engineering

Construcción de las mismas 28 features definidas en el entrenamiento.
Las estadísticas que dependen de train (frecuencia de artista/álbum, lista
de géneros observados) se calculan exclusivamente sobre `train` y se aplican
de forma consistente sobre `test`, evitando cualquier filtración de
información del conjunto de test hacia el proceso de entrenamiento.


In [19]:
def build_features(df_train, df_test):
    """
    Construye las features para train y test de forma consistente.
    Todas las estadísticas se calculan a partir de df_train únicamente.
    """
    train_fe = df_train.copy()
    test_fe  = df_test.copy()

    for df in [train_fe, test_fe]:
        df['release_date_str'] = df['track_album_release_date'].astype(str)
        df['release_year_only'] = df['release_date_str'].str.len() == 4
        df['release_date_missing'] = df['release_date_str'].isin(['nan','','None','0'])
        year_extracted = df['release_date_str'].str[:4]
        df['release_year'] = pd.to_numeric(year_extracted, errors='coerce')
        df['release_year'] = df['release_year'].where(
            df['release_year'].between(1900, 2026), other=np.nan)
        df['release_age_days'] = (2026 - df['release_year']) * 365
        df['release_jan1'] = df['release_date_str'].str.endswith('-01-01')

    for df in [train_fe, test_fe]:
        df['energy_x_loudness']         = df['energy'] * df['loudness'].abs()
        df['valence_x_danceability']    = df['valence'] * df['danceability']
        df['energy_minus_acousticness'] = df['energy'] - df['acousticness']
        df['duration_min']              = df['duration_ms'] / 60000
        df['loudness_norm']       = (df['loudness'] + 60) / 60
        df['instrumental_energy'] = df['instrumentalness'] * df['energy']
        df['dance_energy']        = df['danceability'] * df['energy']
        df['acoustic_valence']    = df['acousticness'] * df['valence']

    artist_count = df_train['track_artist'].value_counts().to_dict()
    train_fe['artist_freq'] = train_fe['track_artist'].map(artist_count).fillna(1)
    test_fe['artist_freq']  = test_fe['track_artist'].map(artist_count).fillna(0)

    album_count = df_train['track_album_id'].value_counts().to_dict()
    train_fe['album_freq'] = train_fe['track_album_id'].map(album_count).fillna(1)
    test_fe['album_freq']  = test_fe['track_album_id'].map(album_count).fillna(0)

    genres_in_train = set(df_train['playlist_genre'].unique())
    for df in [train_fe, test_fe]:
        df['genre_seen_in_train'] = df['playlist_genre'].isin(genres_in_train).astype(int)

    return train_fe, test_fe


train_fe, test_fe = build_features(train, test)

AUDIO_COLS = ['danceability','energy','key','loudness','mode','speechiness',
              'acousticness','instrumentalness','liveness','valence','tempo','duration_ms']
DATE_COLS = ['release_year','release_age_days','release_date_missing',
             'release_year_only','release_jan1']
INTERACTION_COLS = ['energy_x_loudness','valence_x_danceability',
                    'energy_minus_acousticness','duration_min']
FREQ_COLS = ['artist_freq','album_freq']
META_COLS = ['genre_seen_in_train']
NEW_COLS  = ['loudness_norm','instrumental_energy','dance_energy','acoustic_valence']

FEATURES = AUDIO_COLS + DATE_COLS + INTERACTION_COLS + FREQ_COLS + META_COLS + NEW_COLS

for col in ['release_date_missing','release_year_only','release_jan1']:
    train_fe[col] = train_fe[col].astype(int)
    test_fe[col]  = test_fe[col].astype(int)

X      = train_fe[FEATURES].replace([np.inf, -np.inf], np.nan)
X_test = test_fe[FEATURES].replace([np.inf, -np.inf], np.nan)
y      = train_fe['track_popularity'].reset_index(drop=True)

genre_groups = train_fe['playlist_genre'].values
rb_mask_train = genre_groups == 'r&b'
rb_mask_test  = test_fe['playlist_genre'].values == 'r&b'

print(f"Total de features: {len(FEATURES)}")
print(f"Filas de r&b en test: {rb_mask_test.sum()} de {len(test_fe)}")


Total de features: 28
Filas de r&b en test: 524 de 6567


## 4. Imputación de valores faltantes

In [20]:
imp = SimpleImputer(strategy='median')
X_imp      = imp.fit_transform(X)
X_test_imp = imp.transform(X_test)
print("Imputador ajustado sobre train y aplicado sobre test.")


Imputador ajustado sobre train y aplicado sobre test.


## 5. Hiperparámetros del modelo

Idénticos a los definidos en `entrenar_modelo_final.ipynb`.


In [21]:
ET_PARAMS = dict(
    n_estimators=500,
    min_samples_leaf=2,
    max_features=0.7,
    random_state=SEED,
    n_jobs=-1,
)
print(f"Hiperparámetros del modelo: {ET_PARAMS}")


Hiperparámetros del modelo: {'n_estimators': 500, 'min_samples_leaf': 2, 'max_features': 0.7, 'random_state': 42, 'n_jobs': -1}


## 6. Entrenamiento de los dos componentes del modelo final

Se reentrenan el Modelo A (general) y el Modelo B (especializado en r&b)
sobre el 100% del conjunto de entrenamiento, de forma idéntica al notebook
de entrenamiento. Al usar la misma semilla aleatoria y los mismos
hiperparámetros sobre los mismos datos, el resultado es determinístico e
idéntico al modelo documentado en `entrenar_modelo_final.ipynb`.


In [22]:
# Modelo A: ExtraTrees general, entrenado con el 100% del train
model_A = ExtraTreesRegressor(**ET_PARAMS)
model_A.fit(X_imp, y)
print("Modelo A entrenado.")

# Modelo B: ExtraTrees especializado, entrenado solo con r&b
X_rb = X_imp[rb_mask_train]
y_rb = y[rb_mask_train].reset_index(drop=True)
model_B = ExtraTreesRegressor(**ET_PARAMS)
model_B.fit(X_rb, y_rb)
print("Modelo B entrenado.")


Modelo A entrenado.
Modelo B entrenado.


## 7. Predicción sobre el conjunto de test

- Observaciones de género **r&b**: combinación 70% Modelo B + 30% Modelo A.
- Observaciones de cualquier otro género (en el test real, **EDM**):
  predicción íntegra del Modelo A.


In [23]:
pred_A_test = np.clip(model_A.predict(X_test_imp), 0, 100)

X_test_rb = X_test_imp[rb_mask_test]
pred_B_test_rb = np.clip(model_B.predict(X_test_rb), 0, 100)

pred_final = pred_A_test.copy()
pred_final[rb_mask_test] = 0.7 * pred_B_test_rb + 0.3 * pred_A_test[rb_mask_test]
pred_final = np.clip(pred_final, 0, 100)

print(f"Predicción generada para {len(pred_final)} filas de test.")
print(f"Media: {pred_final.mean():.2f}  |  Desvío: {pred_final.std():.2f}")


Predicción generada para 6567 filas de test.
Media: 38.29  |  Desvío: 13.12


## 8. Generación del archivo de submission

In [24]:
submission = pd.DataFrame({
    'ID': test.index,
    'track_popularity': np.round(pred_final, 2)
})

# Verificaciones de formato exigidas por la competencia
assert len(submission) == len(test), "La cantidad de filas no coincide con el test"
assert submission['track_popularity'].isna().sum() == 0, "Hay valores faltantes en la predicción"
assert submission['track_popularity'].between(0, 100).all(), "Hay predicciones fuera de rango [0,100]"

submission.to_csv('submission.csv', index=False)

print("✓ Archivo 'submission_github.csv' generado correctamente.")
print()
print(submission.head(10).to_string(index=False))


✓ Archivo 'submission_github.csv' generado correctamente.

   ID  track_popularity
26266             28.12
26267             37.39
26268             31.72
26269             27.20
26270             31.15
26271             27.62
26272             24.93
26273             32.52
26274             28.84
26275             29.49
